# im gonna preen !!!
Initial stuff for BERTopic

### Initial Python Data Filtering for converting the data into Spark

Choose a data path below if you want to and put data into spark with filtering

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
  SparkSession.builder
  .appName("Phase2")
  .config("spark.driver.memory", "8g")
  .config("spark.executor.memory", "8g")
  .config("spark.sql.debug.maxToStringFields", "100")
  .getOrCreate()
)

review_df = spark.read.parquet("/home/cs179g/repo/data/clean_reviews.parquet")
row_count = review_df.count()
print("Number of rows: " + str(row_count))

26/11/20 20:48:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Number of rows: 1787844


In [4]:
review_df = review_df.withColumn('wordCount', F.size(F.split(F.col('review'), ' ')))
averageWord = review_df.select(F.mean('wordCount')).collect()
medianWord = review_df.select(F.median('wordCount')).collect()
print("Average Word Count: " + str(averageWord) + "\nMedian Word Count: " + str(medianWord))

Average Word Count: [Row(avg(wordCount)=13.328105248556362)]
Median Word Count: [Row(median(wordCount)=4.0)]


In [22]:
small_df = review_df.sample(withReplacement=False, fraction=0.15, seed=3)
row_count = small_df.count()
print("Number of rows: " + str(row_count))

Number of rows: 267876


In [23]:
small_df.head(10)

[Row(recommendationid='26635440', last_played=1600896142, author_num_reviews=2, playtime_forever=23012, playtime_at_review=19158, playtime_last_two_weeks=0, steamid='76561198083061306', language='english', comment_count=0, steam_purchase=True, received_for_free=False, refunded=False, review='5/7', voted_up=True, votes_up=0, votes_funny=0, weighted_vote_score=0.5, written_during_early_access=False, timestamp_created=1479330057, timestamp_updated=1479330057, has_hardware=False, adapter_description=None, cpu_name=None, os=None, system_ram=None, vram_size=None, clean_text='5/7', raw_word_count=1, post_review_playtime=3854, timestamp_formatted='2016-11-16'),
 Row(recommendationid='26635036', last_played=1793490067, author_num_reviews=17, playtime_forever=190334, playtime_at_review=100127, playtime_last_two_weeks=0, steamid='76561198329673016', language='english', comment_count=1, steam_purchase=True, received_for_free=False, refunded=False, review='.', voted_up=True, votes_up=2, votes_funny

Compile an array of reviews for the BERTopic model

In [24]:
from pyspark.sql.functions import collect_list
from pyspark.sql.functions import lit

reviewArr=(small_df.groupBy(lit(1)).agg(collect_list('review'))).first()['collect_list(review)']

### Initialize bertopic stuff

In [ ]:
from bertopic import BERTopic
from bertopic import __init__
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from umap import UMAP

umap_model = UMAP(n_neighbors=50, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(reviewArr,
                                   show_progress_bar=True,
                                   normalize_embeddings=True,
                                   batch_size=256
                                   )

vectorizer_model = CountVectorizer(stop_words="english")
topic_model = BERTopic(
  vectorizer_model=vectorizer_model,
  low_memory=True,
  verbose=True,
  umap_model=umap_model,
  min_topic_size=300,
  nr_topics=25
)
topics, probs = topic_model.fit_transform(reviewArr, embeddings)

Batches: 100%|██████████| 1047/1047 [48:24<00:00,  2.77s/it] 
2026-11-19 18:33:36,399 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-19 19:36:58,564 - BERTopic - Dimensionality - Completed ✓
2026-11-19 19:36:58,593 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-19 19:37:15,106 - BERTopic - Cluster - Completed ✓
2026-11-19 19:37:15,108 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-11-19 19:37:20,264 - BERTopic - Representation - Completed ✓
2026-11-19 19:37:20,271 - BERTopic - Topic reduction - Reducing number of topics
2026-11-19 19:37:20,973 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-19 19:37:24,986 - BERTopic - Representation - Completed ✓
2026-11-19 19:37:25,022 - BERTopic - Topic reduction - Reduced number of topics from 2901 to 25


In [26]:
topic_model.get_topic_info().head(25)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,79776,-1_game_play_fun_like,"[game, play, fun, like, good, just, dont, csgo...",[A very good and fun game to play with friends...
1,0,103540,0_game_good_best_fun,"[game, good, best, fun, great, play, nice, che...","[Top game, it is a game, game]"
2,1,43770,1_yes_good_nice_ok,"[yes, good, nice, ok, 333, cool, lmao, awesome...","[yes good, yes nice with nice yes\n, Very nice..."
3,2,22405,2_gg_gud_ez_noice,"[gg, gud, ez, noice, chickens, rep, gay, pog, ...","[GG, Game Is Good 😂 GG, good game gg]"
4,3,6793,3_russian_cyka_russians_blyat,"[russian, cyka, russians, blyat, rush, learn, ...","[CYKA BLYAT RUSH B!!!!!!!, RUSH B CYKA BLYAT, ..."
5,4,3925,4_que_jogo_veri_si,"[que, jogo, veri, si, muito, joc, bom, sa, la,...","[Jogo muito bom., jogo muito bom, Jogo muito bom]"
6,5,2419,5_cancer_addictive_stroke_life,"[cancer, addictive, stroke, life, syndrome, ad...","[cancer, cancer game, cancer game :(]"
7,6,1550,6_position_tr_spoilerspoiler_cg0d,"[position, tr, spoilerspoiler, cg0d, td, table...",[[table]\n[tr]\n\t[td]⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀...
8,7,1086,7_pew_boom_bang_da,"[pew, boom, bang, da, die, ayy, wayoh, gang, i...","[pew pew pew, Very pew pew\n, pew pew]"
9,8,594,8_httpssteamcommunitycomidjl13inventory7302388...,[httpssteamcommunitycomidjl13inventory73023882...,"[I did this because of my level up on steam, I..."


In [27]:
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
topic_model.save("/home/cs179g/repo/data/BERTopicModel", serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

In [28]:
import numpy as np
np.save("/home/cs179g/repo/data/embeddings.npy", embeddings)

In [33]:
topic_model.get_representative_docs(14)

['Source 2 when???', 'source 2 when...', 'Source 2']